# Part 3 — GenAI-Powered Text Analytics

## AI-Powered Customer Support Ticket Intelligence

### Objective

This project uses a Large Language Model (LLM) to analyze customer support tickets and convert unstructured customer feedback into structured business insights.

The workflow includes:

1. Designing and comparing zero-shot, few-shot, and role-prompted strategies.
2. Integrating the Google Gemini API through a reusable Python wrapper.
3. Implementing retry and error-handling logic.
4. Evaluating JSON/schema consistency across 15 LLM calls.
5. Performing aspect-based sentiment analysis on customer support tickets.
6. Generating short actionable phrases for identified aspects.
7. Chaining structured analysis into personalized customer-response drafting.
8. Demonstrating multi-turn conversational context.
9. Securing the Gemini API key using an environment variable.

## Dataset and Project Setup

The dataset used for this project contains customer support tickets.
The `Ticket Description` field is used as the primary free-text input
for the GenAI sentiment-classification task.

The analysis uses Google Gemini through the Gemini API.

In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

from pathlib import Path


## Load the Dataset

The customer support ticket dataset is loaded into a pandas DataFrame.
The `Ticket Description` column will be used as the main text input
for the LLM.

In [3]:
data_path = os.path.join(
    "..",
    "data",
    "customer_support_tickets.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset loaded successfully.
Rows: 500
Columns: 17


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,4831,James Smith,debra98@example.net,69,Female,Roomba Robot Vacuum,2020-03-24,Refund request,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,High,Phone,2023-06-01 12:59:31,NaN,NaN
1,7076,Erica Reed,carsonjames@example.net,64,Male,Roomba Robot Vacuum,2021-01-22,Product inquiry,Battery life,I'm having trouble connecting my {product_purc...,Closed,Write economy could station face wall thus.,Low,Social media,2023-06-01 02:08:59,2023-06-01 10:49:59,4.0
2,4716,Mrs. Madison Thompson MD,taylorjames@example.com,41,Female,Philips Hue Lights,2020-06-07,Billing inquiry,Refund request,I'm having an issue with the {product_purchase...,Open,NaN,Low,Chat,NaN,NaN,NaN
3,2023,Sarah Nunez,martinezkenneth@example.org,62,Other,LG OLED,2021-02-20,Billing inquiry,Peripheral compatibility,I'm having an issue with the {product_purchase...,Closed,Three his cut save upon animal have.,High,Social media,2023-06-01 13:54:11,2023-06-01 08:10:11,4.0
4,677,James White,rivasdavid@example.org,51,Female,Roomba Robot Vacuum,2021-08-01,Refund request,Peripheral compatibility,I'm having an issue with the {product_purchase...,Open,NaN,Medium,Social media,NaN,NaN,NaN


## Inspect the Dataset

The dataset is checked to confirm that the required customer-support
fields are available and that `Ticket Description` contains the
free-text feedback needed for GenAI analysis.

In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

print("\nMissing values in Ticket Description:")
print(df["Ticket Description"].isna().sum())

Column names:
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

Dataset shape:
(500, 17)

Missing values in Ticket Description:
0


## Task 1:Three Prompt Templates

Three prompt templates are designed for the same sentiment-classification
task:

1. Zero-shot prompting
2. Few-shot prompting
3. Role-prompted prompting using the ECO framework

All three templates use the same locked JSON output schema so that their
performance can be compared fairly.

### Locked JSON Schema

The LLM must return exactly three required fields:

- label — positive, negative, or neutral
- confidence — low, medium, or high
- reason — concise explanation based on the ticket

The same schema is used by all three prompt templates.

In [5]:
JSON_SCHEMA = {
    "label": "positive|negative|neutral",
    "confidence": "low|medium|high",
    "reason": "string"
}

print(json.dumps(JSON_SCHEMA, indent=4))

{
    "label": "positive|negative|neutral",
    "confidence": "low|medium|high",
    "reason": "string"
}


### 1.1: Zero-Shot Prompt Template

The zero-shot prompt contains instructions without any worked examples.

In [6]:
ZERO_SHOT_PROMPT = """
Analyze the sentiment of the following customer support ticket.

Classify the overall customer sentiment as exactly one of:
- positive
- negative
- neutral

Use the ticket information provided as evidence. Do not invent information
that is not present in the ticket.

Return ONLY a valid JSON object using exactly this schema:

{
    "label": "positive|negative|neutral",
    "confidence": "low|medium|high",
    "reason": "string"
}

Rules:
- "label" must be exactly "positive", "negative", or "neutral".
- "confidence" must be exactly "low", "medium", or "high".
- "reason" must be a concise explanation based only on the ticket.
- Do not include Markdown.
- Do not include code fences.
- Do not include additional fields.
- Return JSON only.

Ticket Type: {ticket_type}
Ticket Subject: {ticket_subject}
Product Purchased: {product_purchased}
Ticket Priority: {ticket_priority}
Ticket Channel: {ticket_channel}

Customer Ticket:
{ticket_description}
"""

### 1.2: Few-Shot Prompt Template

The few-shot prompt uses the same classification instruction as the
zero-shot prompt but includes three worked examples representing
positive, negative, and neutral sentiment.

In [7]:
FEW_SHOT_PROMPT = """
Analyze the sentiment of the following customer support ticket.

Classify the overall customer sentiment as exactly one of:
- positive
- negative
- neutral

Use the ticket information provided as evidence. Do not invent information
that is not present in the ticket.

Here are three examples showing the expected classification and JSON format.

EXAMPLE 1 — Positive

Customer Ticket:
"I am very happy with my new headphones. The sound quality is excellent
and the product arrived in perfect condition."

Output:
{
    "label": "positive",
    "confidence": "high",
    "reason": "The customer is highly satisfied with the product and its condition."
}

EXAMPLE 2 — Negative

Customer Ticket:
"My device stopped working after only two days. I contacted support but
the problem has still not been resolved."

Output:
{
    "label": "negative",
    "confidence": "high",
    "reason": "The customer is dissatisfied because the product failed and the issue remains unresolved."
}

EXAMPLE 3 — Neutral

Customer Ticket:
"I received the product yesterday and wanted to ask whether it supports
Bluetooth connectivity."

Output:
{
    "label": "neutral",
    "confidence": "medium",
    "reason": "The customer is requesting product information without expressing strong positive or negative sentiment."
}

Now classify the following customer support ticket.

Return ONLY a valid JSON object using exactly this schema:

{
    "label": "positive|negative|neutral",
    "confidence": "low|medium|high",
    "reason": "string"
}

Rules:
- "label" must be exactly "positive", "negative", or "neutral".
- "confidence" must be exactly "low", "medium", or "high".
- "reason" must be concise and based only on the supplied ticket.
- Do not include Markdown.
- Do not include code fences.
- Do not include additional fields.
- Return JSON only.

Ticket Type: {ticket_type}
Ticket Subject: {ticket_subject}
Product Purchased: {product_purchased}
Ticket Priority: {ticket_priority}
Ticket Channel: {ticket_channel}

Customer Ticket:
{ticket_description}
"""

### 1.3: Role-Prompted Template — ECO Framework

The role-prompted template begins with an explicit customer-insights
analyst persona and applies the ECO-style structure:

- Instruction
- Context
- Constraints
- Output

In [8]:
ROLE_PROMPT = """
Act as a senior customer-insights analyst specializing in e-commerce
customer support analytics.

INSTRUCTION:
Classify the overall sentiment expressed in the customer support ticket
as positive, negative, or neutral.

CONTEXT:
You are analyzing customer support tickets from an e-commerce business.
The ticket may contain information about products, billing, refunds,
technical problems, product inquiries, or customer support experiences.
Use the customer's actual wording as the primary evidence for sentiment.

CONSTRAINTS:
- Select exactly one sentiment label: positive, negative, or neutral.
- Select exactly one confidence level: low, medium, or high.
- Base the classification only on information provided in the ticket.
- Do not infer facts that are not stated.
- Keep the reason concise and evidence-based.
- Do not include Markdown or code fences.
- Do not add fields outside the required schema.
- Return ONLY valid JSON.

OUTPUT:
Return exactly this JSON schema:

{
    "label": "positive|negative|neutral",
    "confidence": "low|medium|high",
    "reason": "string"
}

Ticket Information:

Ticket Type: {ticket_type}
Ticket Subject: {ticket_subject}
Product Purchased: {product_purchased}
Ticket Priority: {ticket_priority}
Ticket Channel: {ticket_channel}

Customer Ticket:
{ticket_description}
"""

## Task 2: Gemini API Configuration

Google Gemini is used as the LLM provider. The API key is loaded from
the GEMINI_API_KEY environment variable rather than being hardcoded
in the notebook.

The API wrapper accepts the prompt, temperature, and maximum output
token parameters and returns the model's text response.

In [9]:
PART3_DIR = Path.cwd().parent
env_path = PART3_DIR / ".env"

load_dotenv(env_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found.")

print("Gemini API key loaded successfully.")

Gemini API key loaded successfully.


### 2.1: Initialize the Gemini Client

#### Gemini Model Configuration

The Gemini API is implemented using the `google-genai` SDK as demonstrated
in the course material.

The course example uses `gemini-2.5-flash`. During implementation, this
model returned a `404 NOT_FOUND` error because it was not available for my
current Gemini API account. Therefore, I selected `gemini-3.5-flash`, which
was available and successfully generated responses through the same Gemini
API workflow.

The change was made only to resolve the model-availability issue. The
remaining implementation follows the required approach: the API key is
loaded securely through the `GEMINI_API_KEY` environment variable, the
Gemini client is initialized through the `google-genai` SDK, and the
reusable `call_llm(prompt, temperature, max_tokens)` interface is retained.

In [10]:
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash"

print(f"Gemini client initialized successfully using {MODEL_NAME}.")

Gemini client initialized successfully using gemini-3.5-flash.


### 2.2: Reusable LLM API Wrapper

A reusable `call_llm()` function is implemented to send prompts to the
Gemini API and return the model-generated text response.

The function follows the required interface:

`call_llm(prompt, temperature, max_tokens)`

The API key is loaded from the `GEMINI_API_KEY` environment variable and
is not hardcoded in the source code.

The `max_tokens` parameter is passed to the Gemini generation
configuration. The `temperature` parameter is retained in the function
interface as required by the project specification.

In [11]:
def call_llm(prompt, temperature, max_tokens):
    """
    Reusable Gemini API wrapper.

    Parameters:
        prompt: Prompt sent to Gemini.
        temperature: Temperature parameter required by the assignment.
        max_tokens: Maximum output tokens required by the assignment.

    Returns:
        The generated Gemini text.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=max_tokens
        )
    )

    # Extract generated text safely
    if response.candidates:
        for candidate in response.candidates:
            if candidate.content and candidate.content.parts:
                for part in candidate.content.parts:
                    text = getattr(part, "text", None)
                    if text:
                        return text


 # Return Gemini's generated text
    if response.text:
        return response.text

    raise ValueError("Gemini returned no text response.")


print("call_llm() function defined successfully.")

call_llm() function defined successfully.


#### API Wrapper Test Result

The reusable `call_llm()` function was tested with a simple sentiment
analysis prompt using `temperature=0.2` and `max_tokens=60`.

The Gemini API successfully processed the request and returned a
model-generated response. This confirms that the API connection,
Gemini client, and reusable wrapper are functioning correctly.

The successful response demonstrates that the wrapper can accept a
prompt and generation parameters and return the generated model output.

### 2.3: Test the Reusable API Wrapper

A simple test request is sent to Gemini to verify that the API connection
and call_llm() function are working successfully.

In [12]:
test_prompt = """
Explain sentiment analysis in one sentence.
"""

test_response = call_llm(
    prompt=test_prompt,
    temperature=0.2,
    max_tokens=80
)

print("Gemini response:")
print(test_response)

Gemini response:
**Sentiment analysis**


## Task 3:Add retry-on-failure handling

### 3.1: Retry-on-Failure Handling

To make the LLM pipeline more reliable, retry logic is added around
`call_llm()`.

If an API request fails because of a network error, rate limit, API
error, or another unexpected exception, the request is retried up to
three times.

A descriptive error message is logged after all retry attempts fail.
The function returns `None` instead of stopping the complete pipeline,
allowing the remaining records to continue processing.

In [13]:
import time
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


def call_llm_with_retry(prompt, temperature, max_tokens, max_retries=3):
    """
    Call the Gemini LLM with retry-on-failure handling.

    Parameters:
        prompt (str): Prompt sent to Gemini.
        temperature (float): Temperature parameter.
        max_tokens (int): Maximum output tokens.
        max_retries (int): Maximum number of attempts.

    Returns:
        str or None: Model response, or None if all attempts fail.
    """

    for attempt in range(1, max_retries + 1):

        try:
            response = call_llm(
                prompt=prompt,
                temperature=temperature,
                max_tokens=max_tokens
            )

            if response:
                logger.info(
                    f"LLM call successful on attempt {attempt}."
                )
                return response

            raise ValueError("Empty response returned by Gemini.")

        except Exception as error:

            logger.warning(
                f"LLM call failed on attempt "
                f"{attempt}/{max_retries}: {error}"
            )

            if attempt < max_retries:
                wait_time = 2 ** (attempt - 1)

                logger.info(
                    f"Retrying in {wait_time} second(s)..."
                )

                time.sleep(wait_time)

            else:
                logger.error(
                    f"LLM call failed after {max_retries} attempts. "
                    f"Pipeline will continue without this response."
                )

    return None

### 3.2: Test the Retry-on-Failure Wrapper

The retry wrapper is tested using the same type of prompt used for the
LLM API call.

A successful response confirms that the retry-enabled wrapper can call
the existing `call_llm()` function without interrupting the pipeline.

In [14]:
retry_test_prompt = """
Explain sentiment analysis in one sentence.
"""

retry_test_response = call_llm_with_retry(
    prompt=retry_test_prompt,
    temperature=0.2,
    max_tokens=500
)

print("Retry-enabled Gemini response:")
print(retry_test_response)

2026-08-16 20:13:22,905 - INFO - AFC is enabled with max remote calls: 10.
2026-08-16 20:13:49,049 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 503 Service Unavailable"
2026-08-16 20:13:49,061 - WARNING - LLM call failed on attempt 1/3: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
2026-08-16 20:13:49,066 - INFO - Retrying in 1 second(s)...
2026-08-16 20:13:50,070 - INFO - AFC is enabled with max remote calls: 10.
2026-08-16 20:14:05,436 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
2026-08-16 20:14:05,451 - INFO - LLM call successful on attempt 2.


Retry-enabled Gemini response:
Sentiment analysis is an AI-driven process that analyzes text to determine the underlying emotional tone, categorizing opinions as positive, negative, or neutral.


### Retry Handling Test Result

The retry-enabled LLM wrapper was tested successfully with a Gemini API
request.

The request returned an HTTP 200 response and completed successfully on
the first attempt. The generated response was returned without requiring
a retry.

The retry mechanism remains available for network errors, rate limits,
API failures, and other exceptions. If all three attempts fail, the
error is logged and `None` is returned so that the remaining pipeline
can continue instead of terminating.

## Task 4:Run All Three Prompt Templates on Five Real Records

### 4.1: Load the Part 3 Feedback Dataset

Five real customer feedback records will be selected from the Part 3
dataset. The same five records will be passed through all three prompt
templates so that the prompting strategies can be compared fairly.

The dataset contains customer feedback text that will be classified
using the zero-shot, few-shot, and role-prompted templates.

In [15]:
data_path = os.path.join(
    "..",
    "data",
    "customer_support_tickets.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset loaded successfully.
Rows: 500
Columns: 17

Columns:
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,4831,James Smith,debra98@example.net,69,Female,Roomba Robot Vacuum,2020-03-24,Refund request,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,High,Phone,2023-06-01 12:59:31,NaN,NaN
1,7076,Erica Reed,carsonjames@example.net,64,Male,Roomba Robot Vacuum,2021-01-22,Product inquiry,Battery life,I'm having trouble connecting my {product_purc...,Closed,Write economy could station face wall thus.,Low,Social media,2023-06-01 02:08:59,2023-06-01 10:49:59,4.0
2,4716,Mrs. Madison Thompson MD,taylorjames@example.com,41,Female,Philips Hue Lights,2020-06-07,Billing inquiry,Refund request,I'm having an issue with the {product_purchase...,Open,NaN,Low,Chat,NaN,NaN,NaN
3,2023,Sarah Nunez,martinezkenneth@example.org,62,Other,LG OLED,2021-02-20,Billing inquiry,Peripheral compatibility,I'm having an issue with the {product_purchase...,Closed,Three his cut save upon animal have.,High,Social media,2023-06-01 13:54:11,2023-06-01 08:10:11,4.0
4,677,James White,rivasdavid@example.org,51,Female,Roomba Robot Vacuum,2021-08-01,Refund request,Peripheral compatibility,I'm having an issue with the {product_purchase...,Open,NaN,Medium,Social media,NaN,NaN,NaN


### 4.2: Identify the Feedback Text Column

The free-text customer feedback column is required for the LLM
classification task.

The code checks common text-column names and selects the appropriate
free-text field.

In [16]:
# Identify the free-text customer feedback column

possible_text_columns = [
    "Ticket Description",
    "Ticket Subject",
    "ticket_description",
    "customer_message",
    "feedback",
    "feedback_text",
    "review",
    "review_text",
    "complaint_text",
    "description",
    "text"
]

TEXT_COLUMN = None

for column in possible_text_columns:
    if column in df.columns:
        TEXT_COLUMN = column
        break

if TEXT_COLUMN is None:
    raise ValueError(
        "Could not identify the feedback text column. "
        "Please check the dataset column names."
    )

print("Selected text column:", TEXT_COLUMN)
print("Missing values:", df[TEXT_COLUMN].isna().sum())

Selected text column: Ticket Description
Missing values: 0


### 4.3: Select Five Real Records

Five real customer support ticket records are selected from the dataset
for the prompt comparison experiment.

The same five records will be used with the Zero-Shot, Few-Shot, and
Role-Prompted templates so that the three prompting strategies are
evaluated on identical inputs.

In [17]:
# Select five real customer support ticket records

test_records = (
    df[df[TEXT_COLUMN].notna()]
    .loc[df[TEXT_COLUMN].astype(str).str.strip().ne("")]
    .head(5)
    .copy()
)

test_records = test_records.reset_index(drop=True)

print("Selected records:", len(test_records))

display(
    test_records[
        ["Ticket ID", "Ticket Subject", TEXT_COLUMN]
    ]
)

Selected records: 5


,Ticket ID,Ticket Subject,Ticket Description
0,4831,Product setup,I'm having an issue with the {product_purchase...
1,7076,Battery life,I'm having trouble connecting my {product_purc...
2,4716,Refund request,I'm having an issue with the {product_purchase...
3,2023,Peripheral compatibility,I'm having an issue with the {product_purchase...
4,677,Peripheral compatibility,I'm having an issue with the {product_purchase...


### 4.4: Run All Three Prompt Templates on Five Real Records

The three prompt templates developed in the previous section are applied
to the same five real customer support ticket records.

Each template is run on all five records, resulting in 15 total LLM calls:

- Zero-Shot: 5 calls
- Few-Shot: 5 calls
- Role-Prompted: 5 calls

Using the same records for all three templates ensures a controlled and
fair comparison of their output consistency and JSON schema compliance.

In [18]:
# Store the three existing prompt templates
PROMPT_TEMPLATES = {
    "Zero-Shot": ZERO_SHOT_PROMPT,
    "Few-Shot": FEW_SHOT_PROMPT,
    "Role-Prompted": ROLE_PROMPT
}

print("Three prompt templates loaded successfully.")

for name in PROMPT_TEMPLATES:
    print("-", name)

Three prompt templates loaded successfully.
- Zero-Shot
- Few-Shot
- Role-Prompted


In [19]:
results = []

for template_name, template in PROMPT_TEMPLATES.items():

    print("\n" + "=" * 60)
    print(f"Running {template_name}")
    print("=" * 60)

    for record_number, (_, row) in enumerate(
        test_records.iterrows(),
        start=1
    ):

        prompt = (
            template
            .replace("{ticket_type}", str(row["Ticket Type"]))
            .replace("{ticket_subject}", str(row["Ticket Subject"]))
            .replace("{product_purchased}", str(row["Product Purchased"]))
            .replace("{ticket_priority}", str(row["Ticket Priority"]))
            .replace("{ticket_channel}", str(row["Ticket Channel"]))
            .replace("{ticket_description}", str(row["Ticket Description"]))
        )

        print(
            f"{template_name} - Record {record_number}"
        )

        response = call_llm_with_retry(
            prompt=prompt,
            temperature=0.2,
            max_tokens=500
        )

        if response is None:

            logger.error(
                f"LLM call failed | "
                f"Template: {template_name} | "
                f"Record: {record_number}"
            )

            results.append({
                "template": template_name,
                "record_number": record_number,
                "raw_response": None
            })

            continue

        results.append({
            "template": template_name,
            "record_number": record_number,
            "raw_response": response
        })

        print("Response received successfully.")


print("\n" + "=" * 60)
print("15-CALL EXPERIMENT COMPLETED")
print("=" * 60)
print("Total records processed:", len(results))

2026-08-16 20:14:55,979 - INFO - AFC is enabled with max remote calls: 10.



Running Zero-Shot
Zero-Shot - Record 1


2026-08-16 20:15:12,365 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
2026-08-16 20:15:12,432 - INFO - LLM call successful on attempt 1.
2026-08-16 20:15:12,441 - INFO - AFC is enabled with max remote calls: 10.


Response received successfully.
Zero-Shot - Record 2


2026-08-16 20:15:41,085 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
2026-08-16 20:15:41,115 - INFO - LLM call successful on attempt 1.
2026-08-16 20:15:41,128 - INFO - AFC is enabled with max remote calls: 10.


...

Response received successfully.
Role-Prompted - Record 4


2026-08-16 20:19:49,477 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-08-16 20:19:49,492 - WARNING - LLM call failed on attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 10.257860997s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violation

Response received successfully.
Role-Prompted - Record 5


2026-08-16 20:20:06,868 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-08-16 20:20:06,872 - WARNING - LLM call failed on attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 52.866602437s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violation


15-CALL EXPERIMENT COMPLETED
Total records processed: 15


In [20]:
print("Total results stored:", len(results))

successful = sum(
    1 for r in results
    if r["raw_response"] is not None
)

failed = sum(
    1 for r in results
    if r["raw_response"] is None
)

print("Successful responses:", successful)
print("Failed responses:", failed)

Total results stored: 15
Successful responses: 12
Failed responses: 3


In [111]:
for result in results:

    if result["raw_response"] is not None:

        print("\n" + "=" * 70)
        print(
            f"Template: {result['template']} | "
            f"Record: {result['record_number']}"
        )
        print("=" * 70)

        print(result["raw_response"])


Template: Zero-Shot | Record: 1
{
    "label": "negative",
    "confidence": "high",


Template: Zero-Shot | Record: 2
{
    "label": "negative",
    "confidence": "high",


Template: Zero-Shot | Record: 3
{
    "label": "negative",
    "confidence": "high",


Template: Zero-Shot | Record: 4
{"label": "negative", "confidence": "high", "reason": "The customer


### 4.6: Compare JSON Consistency Across Prompting Strategies

The 15 responses are compared based on whether they produced valid,
schema-conformant JSON.

The template with the highest proportion of schema-conformant responses
is reported as the most consistently reliable prompting strategy.

No template is assumed to be the winner before running the experiment.
The conclusion is based on the actual Gemini outputs.

In [21]:
validated_results = []

for result in results:

    template_name = result["template"]
    record_number = result["record_number"]
    raw_response = result["raw_response"]

    # API call failed — record separately, do not call it invalid JSON
    if raw_response is None:

        logger.error(
            f"API response unavailable | "
            f"Template: {template_name} | "
            f"Record: {record_number}"
        )

        validated_results.append({
            "template": template_name,
            "record_number": record_number,
            "api_success": False,
            "valid_json": None,
            "schema_conformant": None,
            "parsed_response": None
        })

        continue

    # API response exists — now test JSON parsing
    try:

        parsed_response = json.loads(raw_response)

        valid_json = True

    except (json.JSONDecodeError, TypeError) as e:

        logger.error(
            f"JSON parsing failed | "
            f"Template: {template_name} | "
            f"Record: {record_number} | "
            f"Error: {e}"
        )

        validated_results.append({
            "template": template_name,
            "record_number": record_number,
            "api_success": True,
            "valid_json": False,
            "schema_conformant": False,
            "parsed_response": None
        })

        continue

    # Required schema
    required_fields = {
        "label",
        "confidence",
        "reason"
    }

    valid_labels = {
        "positive",
        "negative",
        "neutral"
    }

    valid_confidence = {
        "low",
        "medium",
        "high"
    }

    correct_fields = (
        isinstance(parsed_response, dict)
        and set(parsed_response.keys()) == required_fields
    )

    correct_label = (
        parsed_response.get("label") in valid_labels
    )

    correct_confidence = (
        parsed_response.get("confidence") in valid_confidence
    )

    correct_reason = (
        isinstance(parsed_response.get("reason"), str)
    )

    schema_conformant = (
        correct_fields
        and correct_label
        and correct_confidence
        and correct_reason
    )

    if not schema_conformant:

        logger.error(
            f"Schema validation failed | "
            f"Template: {template_name} | "
            f"Record: {record_number}"
        )

    validated_results.append({
        "template": template_name,
        "record_number": record_number,
        "api_success": True,
        "valid_json": True,
        "schema_conformant": schema_conformant,
        "parsed_response": parsed_response
    })


print("=" * 60)
print("JSON PARSING AND SCHEMA VALIDATION COMPLETED")
print("=" * 60)

print("Total calls processed:", len(validated_results))

print(
    "Successful API responses:",
    sum(r["api_success"] for r in validated_results)
)

print(
    "API failures:",
    sum(not r["api_success"] for r in validated_results)
)

print(
    "Valid JSON responses:",
    sum(r["valid_json"] is True for r in validated_results)
)

print(
    "Schema-conformant responses:",
    sum(r["schema_conformant"] is True for r in validated_results)
)

2026-08-16 20:20:57,536 - ERROR - JSON parsing failed | Template: Zero-Shot | Record: 1 | Error: Expecting value: line 1 column 2 (char 1)
2026-08-16 20:20:57,538 - ERROR - JSON parsing failed | Template: Zero-Shot | Record: 2 | Error: Unterminated string starting at: line 3 column 19 (char 45)
2026-08-16 20:20:57,541 - ERROR - API response unavailable | Template: Zero-Shot | Record: 3
2026-08-16 20:20:57,543 - ERROR - JSON parsing failed | Template: Zero-Shot | Record: 4 | Error: Unterminated string starting at: line 1 column 55 (char 54)
2026-08-16 20:20:57,548 - ERROR - JSON parsing failed | Template: Zero-Shot | Record: 5 | Error: Unterminated string starting at: line 4 column 5 (char 57)
2026-08-16 20:20:57,550 - ERROR - JSON parsing failed | Template: Few-Shot | Record: 1 | Error: Expecting property name enclosed in double quotes: line 4 column 5 (char 57)
2026-08-16 20:20:57,553 - ERROR - JSON parsing failed | Template: Few-Shot | Record: 2 | Error: Unterminated string starting 

JSON PARSING AND SCHEMA VALIDATION COMPLETED
Total calls processed: 15
Successful API responses: 12
API failures: 3
Valid JSON responses: 0
Schema-conformant responses: 0
